In [1]:
# load train/test datasets
# get model
# build few-shot prompt
# generate output
# evaluate
# generate report

In [2]:
import sys
sys.path.append("../")

In [3]:
# load train/test datasets
from pathlib import Path
from src.llm.loading import Loader
from src.llm.prompt import system_prompt


loader = Loader(path=Path("../data/data.json"), test_k=0.2, seed=42)
train_dataset, tests = loader.train_test_split()
train_dataset, tests

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(Dataset({
     features: ['input', 'target'],
     num_rows: 39
 }),
 Dataset({
     features: ['input', 'target'],
     num_rows: 10
 }))

In [4]:
# get model
from src.llm.models import Model


model = Model(name="Qwen/Qwen2.5-1.5B-Instruct")
tokenizer = model.tokenizer
model.name

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 11340.85it/s]


'Qwen/Qwen2.5-1.5B-Instruct'

In [5]:
# build few-shot prompt
FEW_SHOT_COUNT = 3
few_shot_examples = [train_dataset[index] for index in range(FEW_SHOT_COUNT)]


def build_prompt(input_text: str) -> str:
    prompt_messages = [
        {
            "role": "system",
            "content": (
                f"{system_prompt.render()}\n\n"
                "Use the training examples below as the formatting reference.\n"
                "Keep all non-sensitive text unchanged.\n"
                "Replace sensitive spans with the same bracketed placeholder labels shown in the examples.\n"
                "Do not invent new labels.\n"
                "Return only the masked text."
            ),
        }
    ]

    for example in few_shot_examples:
        prompt_messages.append({"role": "user", "content": example["input"]})
        prompt_messages.append({"role": "assistant", "content": example["target"]})

    prompt_messages.append({"role": "user", "content": input_text})

    return tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

In [6]:
# generate output

result = []
for test in tests:
    output = model.generate(build_prompt(test["input"]))
    result.append({
        "input": test["input"],
        "prediction": output,
        "target": test["target"],
    })

len(result)

10

In [7]:
# evaluate
from src.evaluating import Evaluator

evaluator = Evaluator(
    predictions=[row["prediction"] for row in result],
    references=[row["target"] for row in result],
)

metrics = evaluator.evaluate()
metrics

{'samples': 10,
 'exact_match': 0.2,
 'masking_recall': 0.5882352941176471,
 'over_masking_rate': 0.3548387096774194,
 'text_preservation': 0.9880843672456576}

In [8]:
# generate report
from src.reporting import write_baseline_report

report_path = write_baseline_report(
    model_name=f"{model.name} (few-shot)",
    metrics=metrics,
    predictions=result,
    output_path="../reports/02_llm_few_shot.md",
    preview_size=len(result)
)

print(f"Few-shot report saved to {report_path}")

preview = result[:3]
for index, row in enumerate(preview, start=1):
    print()
    print(f"Sample {index}")
    print("INPUT:", row["input"])
    print("PREDICTION:", row["prediction"])
    print("TARGET:", row["target"])

Few-shot report saved to ../reports/02_llm_few_shot.md

Sample 1
INPUT: Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com.
PREDICTION: Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].
TARGET: Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].

Sample 2
INPUT: Dear Ms. Garcia, I need to report a lost debit card. My phone number is (555) 789-0123 and my account number is 678901.
PREDICTION: Dear [PREFIX], I need to report a lost debit card. My phone number is [(PHONE_NUMBER)] and my account number is [ACCOUNT_NUMBER].
TARGET: Dear [PREFIX], I need to report a lost debit card. My phone number is [PHONE_NUMBER] and my account number is [ACCOUNT_NUMBER].

Sample 3
INPUT: I'd like to know 